# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains mass spectrometry protein analysis metadata and records.

In [ ]:
# Ensure `mlcroissant` is installed (will do nothing if already installed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. All references to dataset entities use their `@id` fields as required by the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets, fields, and their `@id`s for discovery.

We use the Dataset's `metadata` API to enumerate the record sets and their fields. All references use Croissant entity `@id`.

In [ ]:
# List all record sets and their field @id`s
record_sets_info = dataset.metadata.record_sets
if not record_sets_info:
    print("No record sets found in the metadata. Please verify dataset schema.")
else:
    for record_set in record_sets_info:
        print(f"RecordSet name: {record_set.name}\n  @id: {record_set.id}")
        print("  Fields and columns (@id):")
        for field in record_set.fields:
            print(f"    - {field.name}: {field.id}")
        print('-' * 40)

For illustration, let's inspect the first record set's records to view the schema/data structure. (Replace with the appropriate `@id` from the list above.)

In [ ]:
# Find the first record set @id:
record_sets = dataset.metadata.record_sets
if record_sets:
    first_record_set_id = record_sets[0].id
    print(f"Sample records from record set with @id={first_record_set_id}:")
    for idx, rec in enumerate(dataset.records(record_set=first_record_set_id)):
        print(rec)
        if idx >= 2:
            break
else:
    first_record_set_id = None
    print('No record sets found.')

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the `@id` properties from the overview above.

In [ ]:
# List all available record set @id's for processing
record_sets = dataset.metadata.record_sets
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for recset_id in record_set_ids:
    records_iter = dataset.records(record_set=recset_id)
    data = list(records_iter)
    try:
        df = pd.DataFrame(data)
        dataframes[recset_id] = df
        print(f"Loaded DataFrame for record set {recset_id} with columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load DataFrame for record set {recset_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Let's perform some standard EDA operations on the first record set. We'll select a suitable numeric field (by `@id`) and demonstrate filtering, normalization, and optional grouping.

Replace the below field and group ids with values appropriate to your use case as shown in section 2.

In [ ]:
import numpy as np

# Use the first record set DataFrame (if available)
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Columns in DataFrame ({record_set_id}):\n{df.columns.tolist()}")
    # Try to pick a numeric field automatically
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())/filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Try grouping by a likely categorical field (if exists)
        possible_group_fields = [col for col in df.columns if df[col].nunique() < 10 and col != numeric_field_id]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped (mean) data by {group_field}:")
            print(grouped_df.head())
    else:
        print(f"No numeric fields found in record set {record_set_id}.")
else:
    print('No data loaded; cannot run EDA.')

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalization. Requires `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_fields:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=30, color='blue', label=numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.legend()
    plt.show()
    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=30, color='green', label=f"{numeric_field_id}_normalized")
        plt.title(f"Distribution of Normalized {numeric_field_id} (filtered)")
        plt.xlabel(f"Normalized {numeric_field_id}")
        plt.ylabel("Count")
        plt.legend()
        plt.show()

## 6. Conclusion
This notebook demonstrated loading, exploring, and analyzing the FAIR² mass spectrometry protein dataset using the `mlcroissant` library. All dataset entities were referenced by their `@id` per Croissant best practices. You can extend this workflow for further biological and statistical analyses, model building, or data publishing.